In [1]:
import os
import sys
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.optim.lr_scheduler import CosineAnnealingLR

# ---------------------------------------------------------
# 0. Dependencies Setup & Metric Imports
# ---------------------------------------------------------
try:
    import medmnist
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "medmnist", "scikit-learn", "matplotlib", "seaborn", "tqdm"])
    import medmnist

from medmnist import INFO, Evaluator
from sklearn.metrics import (
    precision_score, recall_score, f1_score, confusion_matrix, 
    roc_curve, auc, classification_report
)

# ---------------------------------------------------------
# 1. Exact MedLiT Components (Paper Specification)
# ---------------------------------------------------------
class SwiGLUExpert(nn.Module):
    """SwiGLU Feedforward Block as specified in MedLiT paper."""
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.w1 = nn.Linear(in_dim, hidden_dim, bias=False)
        self.v = nn.Linear(in_dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, in_dim, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.v(x))

class MedLiTSparseMoE(nn.Module):
    """Top-2 Gated MoE Layer with SwiGLU experts and balancing loss."""
    def __init__(self, in_dim, hidden_dim, num_experts, k=2):
        super().__init__()
        self.k = k
        self.num_experts = num_experts
        self.gate = nn.Linear(in_dim, num_experts, bias=False)
        self.experts = nn.ModuleList([SwiGLUExpert(in_dim, hidden_dim) for _ in range(num_experts)])

    def forward(self, x):
        B, N, C = x.shape
        x_flat = x.view(-1, C)
        
        logits = self.gate(x_flat)
        probs = F.softmax(logits, dim=-1)
        topk_weights, topk_indices = torch.topk(probs, self.k, dim=-1)
        
        # Load Balancing Loss calculation
        importance = probs.sum(0)
        tokens_per_expert = torch.bincount(topk_indices.view(-1), minlength=self.num_experts).float()
        l_aux = self.num_experts * torch.sum(importance * tokens_per_expert) / (B * N * self.k)

        out = torch.zeros_like(x_flat)
        for i, expert in enumerate(self.experts):
            mask = (topk_indices == i)
            if mask.any():
                token_idx, k_idx = torch.where(mask)
                expert_out = expert(x_flat[token_idx])
                out[token_idx] += expert_out * topk_weights[token_idx, k_idx].unsqueeze(-1)
                
        return out.view(B, N, C), l_aux

class GroupedQueryAttention(nn.Module):
    """Grouped Query Attention (6 heads split into 3 groups)."""
    def __init__(self, embed_dim, num_heads=6, num_groups=3):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.num_groups = num_groups
        self.head_dim = embed_dim // num_heads
        self.heads_per_group = num_heads // num_groups

        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, self.num_groups * self.head_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, self.num_groups * self.head_dim, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)

    def forward(self, x):
        B, N, C = x.shape
        q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, N, self.num_groups, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, N, self.num_groups, self.head_dim).transpose(1, 2)

        k = k.repeat_interleave(self.heads_per_group, dim=1)
        v = v.repeat_interleave(self.heads_per_group, dim=1)

        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn = F.softmax(scores, dim=-1)
        context = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, N, C)
        return self.out_proj(context)

class MedLiTBlock(nn.Module):
    def __init__(self, embed_dim, ffn_hidden_dim, num_experts):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = GroupedQueryAttention(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.moe = MedLiTSparseMoE(embed_dim, ffn_hidden_dim, num_experts)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        moe_out, l_aux = self.moe(self.ln2(x))
        x = x + moe_out
        return x, l_aux

class MedLiT_Seed(nn.Module):
    """
    Official MedLiT-seed Architecture (2.11M parameters):
    - Embedding Dimension: 216
    - 3 Stages (9 layers total)
    - Expert counts per stage: [3, 4, 5]
    - Progressive FFN Scaling: [81, 54, 27]
    """
    def __init__(self, num_classes=9, img_size=224, patch_size=16, embed_dim=216):
        super().__init__()
        self.patch_proj = nn.Conv2d(3, embed_dim, kernel_size=patch_size, stride=patch_size)
        num_patches = (img_size // patch_size) ** 2
        
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))

        stage_experts = [3, 4, 5]
        ffn_dims = [81, 54, 27]
        
        self.layers = nn.ModuleList()
        for stage_idx in range(3):
            for _ in range(3):
                self.layers.append(
                    MedLiTBlock(
                        embed_dim=embed_dim, 
                        ffn_hidden_dim=ffn_dims[stage_idx], 
                        num_experts=stage_experts[stage_idx]
                    )
                )

        self.ln = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        
        # Proper parameter initialization for Vision Transformer
        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.xavier_uniform_(self.patch_proj.weight)
        if self.patch_proj.bias is not None:
            nn.init.zeros_(self.patch_proj.bias)
        nn.init.normal_(self.head.weight, std=0.02)
        if self.head.bias is not None:
            nn.init.zeros_(self.head.bias)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_proj(x).flatten(2).transpose(1, 2)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1) + self.pos_embed

        total_aux_loss = 0.0
        for layer in self.layers:
            x, l_aux = layer(x)
            total_aux_loss += l_aux

        x = self.ln(x[:, 0])
        logits = self.head(x)
        return logits, total_aux_loss

class MedLiTFocalLoss(nn.Module):
    def __init__(self, tau=0.8, gamma=2.0):
        super().__init__()
        self.tau = tau
        self.gamma = gamma

    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=-1)
        p_t = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        
        mask = (p_t < self.tau).float()
        focal_weight = (1.0 - p_t) ** self.gamma
        
        # Mean across the batch dimension
        loss = (mask * focal_weight * ce_loss).mean()
        return loss

# Helper function for FGSM perturbation generation
def fgsm_attack(image, epsilon, data_grad):
    sign_data_grad = data_grad.sign()
    perturbed_image = image + epsilon * sign_data_grad
    perturbed_image = torch.clamp(perturbed_image, -1.0, 1.0)
    return perturbed_image

# ---------------------------------------------------------
# 2. Dataset Loader for Kaggle Path
# ---------------------------------------------------------
class LocalPathMNISTDataset(Dataset):
    def __init__(self, npz_path, split='train', transform=None):
        npz_file = np.load(npz_path)
        self.images = npz_file[f'{split}_images']
        self.labels = npz_file[f'{split}_labels']
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.fromarray(self.images[idx])
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

data_npz_path = "pathmnist.npz"
data_flag = 'pathmnist'
info = INFO[data_flag]
num_classes = len(info['label'])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Loading data from: {data_npz_path}")
print(f"Executing on Device: {device}")

# Improved augmentations critical for histology training
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

train_dataset = LocalPathMNISTDataset(data_npz_path, split='train', transform=train_transform)
val_dataset = LocalPathMNISTDataset(data_npz_path, split='val', transform=eval_transform)
test_dataset = LocalPathMNISTDataset(data_npz_path, split='test', transform=eval_transform)

BATCH_SIZE = 256
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# ---------------------------------------------------------
# 3. Model & Training Instantiation
# ---------------------------------------------------------
model = MedLiT_Seed(num_classes=num_classes, img_size=224, embed_dim=216).to(device)
criterion = MedLiTFocalLoss(tau=0.8, gamma=2.0)
optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-2)
epochs = 100
scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)

# Training parameters for adversarial robustness
TRAIN_EPSILON = [0.05] #[0.0, 0.01, 0.03, 0.05, 0.1]

# ---------------------------------------------------------
# 4. FGSM Adversarial Training & Validation Loops
# ---------------------------------------------------------
def train_one_epoch_adv(model, loader, criterion, optimizer, device, epoch, epsilon=0.05):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc=f"Epoch {epoch+1:03d} [AdvTrain]", leave=False, disable=True)
    num_layers = len(model.layers)
    
    for inputs, targets in pbar:
        inputs = inputs.to(device)
        targets = targets.squeeze().long().to(device)
        
        # Step 1: Generate adversarial samples dynamically on input batch
        inputs.requires_grad = True
        outputs_clean, _ = model(inputs)
        loss_clean_init = criterion(outputs_clean, targets)
        
        model.zero_grad()
        loss_clean_init.backward()
        data_grad = inputs.grad.data
        
        # Create perturbed adversarial inputs
        inputs_adv = fgsm_attack(inputs.detach(), epsilon, data_grad)
        
        # Step 2: Forward pass using composite loss (50% clean + 50% adversarial)
        optimizer.zero_grad()
        outputs_clean, aux_clean = model(inputs.detach())
        outputs_adv, aux_adv = model(inputs_adv)
        
        loss_clean = criterion(outputs_clean, targets) + 0.01 * (aux_clean / num_layers)
        loss_adv = criterion(outputs_adv, targets) + 0.01 * (aux_adv / num_layers)
        
        total_step_loss = 0.5 * loss_clean + 0.5 * loss_adv
        total_step_loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        batch_size = inputs.size(0)
        total_loss += total_step_loss.item() * batch_size
        
        _, predicted = outputs_adv.max(1)  # Track accuracy on adversarial inputs
        total += batch_size
        correct += predicted.eq(targets).sum().item()
        
        pbar.set_postfix({"Loss": f"{total_loss / total:.4f}", "AdvAcc": f"{(correct / total) * 100:.2f}%"})

    return total_loss / total, correct / total

def validate(model, loader, criterion, device, epoch):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc=f"Epoch {epoch+1:03d} [Val]  ", leave=False, disable=True)
    num_layers = len(model.layers)
    
    with torch.no_grad():
        for inputs, targets in pbar:
            inputs = inputs.to(device)
            targets = targets.squeeze().long().to(device)
            outputs, aux_loss = model(inputs)
            loss = criterion(outputs, targets) + 0.01 * (aux_loss / num_layers)

            total_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            pbar.set_postfix({"Loss": f"{total_loss/total:.4f}", "Acc": f"{(correct/total)*100:.2f}%"})

    return total_loss / total, correct / total

train_losses, val_losses = [], []
train_accs, val_accs = [], []

print(f"\n--- Starting FGSM Adversarial Training (Epsilon Schedule) ---")
for epoch in range(epochs):
    tr_loss, tr_acc = train_one_epoch_adv(model, train_loader, criterion, optimizer, device, epoch, epsilon=random.choice(TRAIN_EPSILON))
    val_loss, val_acc = validate(model, val_loader, criterion, device, epoch)
    scheduler.step()

    train_losses.append(tr_loss)
    val_losses.append(val_loss)
    train_accs.append(tr_acc)
    val_accs.append(val_acc)

    print(f"Epoch [{epoch+1:03d}/{epochs}] | Train Loss: {tr_loss:.4f} AdvAcc: {tr_acc*100:.2f}% | Val Loss: {val_loss:.4f} Acc: {val_acc*100:.2f}%")

# ---------------------------------------------------------
# 5. Final Evaluation & Report Visualizations
# ---------------------------------------------------------
def generate_report_artifacts(model, loader, device, num_classes, class_labels):
    model.eval()
    y_score, y_true = [], []
    
    with torch.no_grad():
        for inputs, targets in tqdm(loader, desc="Testing"):
            inputs = inputs.to(device)
            outputs, _ = model(inputs)
            outputs = outputs.softmax(dim=-1)
            y_score.append(outputs.cpu().numpy())
            y_true.append(targets.squeeze().numpy())
            
    y_score = np.concatenate(y_score, axis=0)
    y_true = np.concatenate(y_true, axis=0)
    y_pred = np.argmax(y_score, axis=1)

    data_dir = os.path.dirname(os.path.abspath(data_npz_path))
    evaluator = Evaluator(data_flag, 'test', root=data_dir)
    medmnist_auc, medmnist_acc = evaluator.evaluate(y_score)
    
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    sensitivity = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

    cm = confusion_matrix(y_true, y_pred)
    specificities = [
        (np.sum(cm) - (cm[i, i] + (np.sum(cm[i, :]) - cm[i, i]) + (np.sum(cm[:, i]) - cm[i, i]))) /
        ((np.sum(cm) - (cm[i, i] + (np.sum(cm[i, :]) - cm[i, i]) + (np.sum(cm[:, i]) - cm[i, i]))) + (np.sum(cm[:, i]) - cm[i, i]))
        for i in range(num_classes)
    ]
    specificity = np.mean(specificities)

    print("\n" + "="*50)
    print("        COMPREHENSIVE TEST EVALUATION REPORT     ")
    print("="*50)
    print(f"Overall Accuracy (OA / ACC) : {medmnist_acc * 100:.2f}%")
    print(f"Area Under Curve (AUC)      : {medmnist_auc:.4f}")
    print(f"Precision (Macro)            : {precision * 100:.2f}%")
    print(f"Sensitivity / Recall (Macro) : {sensitivity * 100:.2f}%")
    print(f"Specificity (Macro)          : {specificity * 100:.2f}%")
    print(f"F1-Score (Macro)             : {f1 * 100:.2f}%")
    print("="*50)
    
    print("\nClass-Wise Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=list(class_labels.values()), digits=4, zero_division=0))

    # Saved Visualizations
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(1, epochs + 1), train_losses, label='Train Loss')
    plt.plot(range(1, epochs + 1), val_losses, label='Val Loss')
    plt.title('Loss Progression')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(range(1, epochs + 1), [a * 100 for a in train_accs], label='Train Acc')
    plt.plot(range(1, epochs + 1), [a * 100 for a in val_accs], label='Val Acc')
    plt.title('Accuracy Progression (%)')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('learning_curves_at.png', dpi=300)
    plt.close()

    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=list(class_labels.values()), yticklabels=list(class_labels.values()))
    plt.title('Test Set Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('confusion_matrix_at.png', dpi=300)
    plt.close()

    plt.figure(figsize=(9, 7))
    for i in range(num_classes):
        fpr, tpr, _ = roc_curve(y_true == i, y_score[:, i])
        plt.plot(fpr, tpr, label=f"{class_labels[str(i)]} (AUC = {auc(fpr, tpr):.3f})")
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Multi-Class ROC Curves (PathMNIST)')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('roc_curves_at.png', dpi=300)
    plt.close()

    print("\nVisualizations saved successfully to Kaggle working directory:")
    print("  1. learning_curves_at.png")
    print("  2. confusion_matrix_at.png")
    print("  3. roc_curves_at.png\n")

print("\n--- Generating Final Evaluation & Visualizations ---")
generate_report_artifacts(model, test_loader, device, num_classes, info['label'])

# ---------------------------------------------------------
# 6. Post-Training FGSM Robustness Benchmark
# ---------------------------------------------------------
def evaluate_fgsm(model, loader, criterion, device, epsilon):
    model.eval()
    correct, total = 0, 0
    num_layers = len(model.layers)
    
    for inputs, targets in tqdm(loader, desc=f"FGSM Eval (eps={epsilon})", leave=False):
        inputs = inputs.to(device)
        targets = targets.squeeze().long().to(device)
        inputs.requires_grad = True
        
        outputs, aux_loss = model(inputs)
        loss = criterion(outputs, targets) + 0.01 * (aux_loss / num_layers)
        
        model.zero_grad()
        loss.backward()
        
        data_grad = inputs.grad.data
        perturbed_inputs = fgsm_attack(inputs.detach(), epsilon, data_grad)
        
        with torch.no_grad():
            outputs_adv, _ = model(perturbed_inputs)
            _, predicted = outputs_adv.max(1)
            
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
    return (correct / total) * 100.0

def run_adversarial_suite(model, val_loader, test_loader, criterion, device, class_labels, epsilons=[0.0, 0.01, 0.03, 0.05, 0.1, 0.15, 0.2]):
    print("\n" + "="*50)
    print("    POST-ADVERSARIAL TRAINING FGSM BENCHMARK     ")
    print("="*50)
    
    val_accuracies, test_accuracies = [], []
    
    for eps in epsilons:
        val_acc = evaluate_fgsm(model, val_loader, criterion, device, eps)
        test_acc = evaluate_fgsm(model, test_loader, criterion, device, eps)
        val_accuracies.append(val_acc)
        test_accuracies.append(test_acc)
        print(f"Epsilon: {eps:<5} | Val Acc: {val_acc:.2f}% | Test Acc: {test_acc:.2f}%")

    # Plot 1: Robustness Curve
    plt.figure(figsize=(9, 6))
    plt.plot(epsilons, val_accuracies, 'o-', color='crimson', linewidth=2, label='Validation Accuracy')
    plt.plot(epsilons, test_accuracies, 's--', color='royalblue', linewidth=2, label='Test Accuracy')
    plt.title('Robustness Curve: Adversarially Trained MedLiT-seed')
    plt.xlabel(r'Adversarial Perturbation Magnitude (Epsilon $\epsilon$)')
    plt.ylabel('Accuracy (%)')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.savefig('fgsm_robustness_curve_at.png', dpi=300)
    plt.close()

    # Plot 2: Visual Comparison Grid
    print("\nGenerating qualitative adversarial visual examples...")
    target_eps = 0.05
    model.eval()
    num_layers = len(model.layers)
    
    sample_inputs, sample_targets = next(iter(test_loader))
    sample_inputs = sample_inputs.to(device)
    sample_targets = sample_targets.squeeze().long().to(device)
    sample_inputs.requires_grad = True

    outputs, aux_loss = model(sample_inputs)
    loss = criterion(outputs, sample_targets) + 0.01 * (aux_loss / num_layers)
    model.zero_grad()
    loss.backward()

    data_grad = sample_inputs.grad.data
    perturbed_inputs = fgsm_attack(sample_inputs.detach(), target_eps, data_grad)

    with torch.no_grad():
        outputs_clean, _ = model(sample_inputs)
        outputs_adv, _ = model(perturbed_inputs)
        
        preds_clean = outputs_clean.argmax(dim=1).cpu().numpy()
        preds_adv = outputs_adv.argmax(dim=1).cpu().numpy()

    inputs_np = (sample_inputs.cpu().detach().numpy().transpose(0, 2, 3, 1) + 1.0) / 2.0
    perturbed_np = (perturbed_inputs.cpu().detach().numpy().transpose(0, 2, 3, 1) + 1.0) / 2.0
    grad_np = (data_grad.sign().cpu().numpy().transpose(0, 2, 3, 1) + 1.0) / 2.0
    targets_np = sample_targets.cpu().numpy()

    num_samples = 4
    fig, axes = plt.subplots(num_samples, 3, figsize=(11, 3.5 * num_samples))
    
    for i in range(num_samples):
        true_lbl = class_labels[str(targets_np[i])]
        clean_lbl = class_labels[str(preds_clean[i])]
        adv_lbl = class_labels[str(preds_adv[i])]

        axes[i, 0].imshow(np.clip(inputs_np[i], 0, 1))
        axes[i, 0].set_title(f"Original Sample\nTrue: {true_lbl}\nPred: {clean_lbl}", color='green' if true_lbl == clean_lbl else 'red')
        axes[i, 0].axis('off')

        axes[i, 1].imshow(np.clip(grad_np[i], 0, 1))
        axes[i, 1].set_title(rf"FGSM Noise Pattern\n($\epsilon = {target_eps}$)")
        axes[i, 1].axis('off')

        axes[i, 2].imshow(np.clip(perturbed_np[i], 0, 1))
        axes[i, 2].set_title(f"Attacked Sample\nTrue: {true_lbl}\nAttacked Pred: {adv_lbl}", color='green' if true_lbl == adv_lbl else 'red')
        axes[i, 2].axis('off')

    plt.tight_layout()
    plt.savefig('fgsm_visual_samples_robust_at.png', dpi=300)
    plt.close()
    print("Saved sample grid visualization: fgsm_visual_samples_robust_at.png\n")

# Run evaluation suite
run_adversarial_suite(model, val_loader, test_loader, criterion, device, info['label'])

Loading data from: pathmnist.npz
Executing on Device: cuda



--- Starting FGSM Adversarial Training (Epsilon Schedule) ---


Epoch [001/100] | Train Loss: 480.8051 AdvAcc: 25.88% | Val Loss: 465.9341 Acc: 29.49%


Epoch [002/100] | Train Loss: 464.9527 AdvAcc: 28.86% | Val Loss: 476.7862 Acc: 31.83%


Epoch [003/100] | Train Loss: 464.7760 AdvAcc: 30.58% | Val Loss: 465.9706 Acc: 33.78%


Epoch [004/100] | Train Loss: 465.1859 AdvAcc: 32.11% | Val Loss: 478.7108 Acc: 34.74%


... (630 outputs left out)

FGSM Eval (eps=0.2): 100%|██████████| 29/29 [00:27<00:00,  1.41it/s]

Epsilon: 0.2   | Val Acc: 32.22% | Test Acc: 47.45%



Generating qualitative adversarial visual examples...


Saved sample grid visualization: fgsm_visual_samples_robust_at.png



In [2]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.optim.lr_scheduler import CosineAnnealingLR

# ---------------------------------------------------------
# 0. Dependencies Setup & Metric Imports
# ---------------------------------------------------------
try:
    import medmnist
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "medmnist", "scikit-learn", "matplotlib", "seaborn", "tqdm"])
    import medmnist

from medmnist import INFO, Evaluator
from sklearn.metrics import (
    precision_score, recall_score, f1_score, confusion_matrix, 
    roc_curve, auc, classification_report
)

# ---------------------------------------------------------
# 1. Exact MedLiT Components (Paper Specification)
# ---------------------------------------------------------
class SwiGLUExpert(nn.Module):
    """SwiGLU Feedforward Block as specified in MedLiT paper."""
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.w1 = nn.Linear(in_dim, hidden_dim, bias=False)
        self.v = nn.Linear(in_dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, in_dim, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.v(x))

class MedLiTSparseMoE(nn.Module):
    """Top-2 Gated MoE Layer with SwiGLU experts and balancing loss."""
    def __init__(self, in_dim, hidden_dim, num_experts, k=2):
        super().__init__()
        self.k = k
        self.num_experts = num_experts
        self.gate = nn.Linear(in_dim, num_experts, bias=False)
        self.experts = nn.ModuleList([SwiGLUExpert(in_dim, hidden_dim) for _ in range(num_experts)])

    def forward(self, x):
        B, N, C = x.shape
        x_flat = x.view(-1, C)
        
        logits = self.gate(x_flat)
        probs = F.softmax(logits, dim=-1)
        topk_weights, topk_indices = torch.topk(probs, self.k, dim=-1)
        
        # Load Balancing Loss calculation
        importance = probs.sum(0)
        tokens_per_expert = torch.bincount(topk_indices.view(-1), minlength=self.num_experts).float()
        l_aux = self.num_experts * torch.sum(importance * tokens_per_expert) / (B * N * self.k)

        out = torch.zeros_like(x_flat)
        for i, expert in enumerate(self.experts):
            mask = (topk_indices == i)
            if mask.any():
                token_idx, k_idx = torch.where(mask)
                expert_out = expert(x_flat[token_idx])
                out[token_idx] += expert_out * topk_weights[token_idx, k_idx].unsqueeze(-1)
                
        return out.view(B, N, C), l_aux

class GroupedQueryAttention(nn.Module):
    """Grouped Query Attention (6 heads split into 3 groups)."""
    def __init__(self, embed_dim, num_heads=6, num_groups=3):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.num_groups = num_groups
        self.head_dim = embed_dim // num_heads
        self.heads_per_group = num_heads // num_groups

        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, self.num_groups * self.head_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, self.num_groups * self.head_dim, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)

    def forward(self, x):
        B, N, C = x.shape
        q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, N, self.num_groups, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, N, self.num_groups, self.head_dim).transpose(1, 2)

        k = k.repeat_interleave(self.heads_per_group, dim=1)
        v = v.repeat_interleave(self.heads_per_group, dim=1)

        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn = F.softmax(scores, dim=-1)
        context = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, N, C)
        return self.out_proj(context)

class MedLiTBlock(nn.Module):
    def __init__(self, embed_dim, ffn_hidden_dim, num_experts):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = GroupedQueryAttention(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.moe = MedLiTSparseMoE(embed_dim, ffn_hidden_dim, num_experts)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        moe_out, l_aux = self.moe(self.ln2(x))
        x = x + moe_out
        return x, l_aux

class MedLiT_Seed(nn.Module):
    """
    Official MedLiT-seed Architecture (2.11M parameters):
    - Embedding Dimension: 216
    - 3 Stages (9 layers total)
    - Expert counts per stage: [3, 4, 5]
    - Progressive FFN Scaling: [81, 54, 27]
    """
    def __init__(self, num_classes=9, img_size=224, patch_size=16, embed_dim=216):
        super().__init__()
        self.patch_proj = nn.Conv2d(3, embed_dim, kernel_size=patch_size, stride=patch_size)
        num_patches = (img_size // patch_size) ** 2
        
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))

        stage_experts = [3, 4, 5]
        ffn_dims = [81, 54, 27]
        
        self.layers = nn.ModuleList()
        for stage_idx in range(3):
            for _ in range(3):
                self.layers.append(
                    MedLiTBlock(
                        embed_dim=embed_dim, 
                        ffn_hidden_dim=ffn_dims[stage_idx], 
                        num_experts=stage_experts[stage_idx]
                    )
                )

        self.ln = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        
        # Proper parameter initialization for Vision Transformer
        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.xavier_uniform_(self.patch_proj.weight)
        if self.patch_proj.bias is not None:
            nn.init.zeros_(self.patch_proj.bias)
        nn.init.normal_(self.head.weight, std=0.02)
        if self.head.bias is not None:
            nn.init.zeros_(self.head.bias)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_proj(x).flatten(2).transpose(1, 2)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1) + self.pos_embed

        total_aux_loss = 0.0
        for layer in self.layers:
            x, l_aux = layer(x)
            total_aux_loss += l_aux

        x = self.ln(x[:, 0])
        logits = self.head(x)
        return logits, total_aux_loss

class MedLiTFocalLoss(nn.Module):
    def __init__(self, tau=0.8, gamma=2.0):
        super().__init__()
        self.tau = tau
        self.gamma = gamma

    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=-1)
        p_t = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        
        mask = (p_t < self.tau).float()
        focal_weight = (1.0 - p_t) ** self.gamma
        
        loss = (mask * focal_weight * ce_loss).mean()
        return loss

# ---------------------------------------------------------
# 2. Dataset Loader for Kaggle Path
# ---------------------------------------------------------
class LocalPathMNISTDataset(Dataset):
    def __init__(self, npz_path, split='train', transform=None):
        npz_file = np.load(npz_path)
        self.images = npz_file[f'{split}_images']
        self.labels = npz_file[f'{split}_labels']
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.fromarray(self.images[idx])
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

data_npz_path = "pathmnist.npz"
data_flag = 'pathmnist'
info = INFO[data_flag]
num_classes = len(info['label'])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Loading data from: {data_npz_path}")
print(f"Executing on Device: {device}")

# Improved augmentations critical for histology training from scratch
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

train_dataset = LocalPathMNISTDataset(data_npz_path, split='train', transform=train_transform)
val_dataset = LocalPathMNISTDataset(data_npz_path, split='val', transform=eval_transform)
test_dataset = LocalPathMNISTDataset(data_npz_path, split='test', transform=eval_transform)

BATCH_SIZE = 256  # Adjusted batch size for stable MoE gradient updates
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# ---------------------------------------------------------
# 3. Model & Training Instantiation
# ---------------------------------------------------------
model = MedLiT_Seed(num_classes=num_classes, img_size=224, embed_dim=216).to(device)
criterion = MedLiTFocalLoss(tau=0.8, gamma=2.0)
optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-2)
epochs = 100
scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)

# ---------------------------------------------------------
# 4. Training & Validation Loops
# ---------------------------------------------------------
def train_one_epoch(model, loader, criterion, optimizer, device, epoch):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc=f"Epoch {epoch+1:03d} [Train]", leave=False, disable=True)
    
    num_layers = len(model.layers)
    
    for inputs, targets in pbar:
        inputs = inputs.to(device)
        targets = targets.squeeze().long().to(device)
        
        optimizer.zero_grad()
        outputs, aux_loss = model(inputs)
        
        cls_loss = criterion(outputs, targets)
        scaled_aux_loss = (aux_loss / num_layers) * 0.01
        
        loss = cls_loss + scaled_aux_loss
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        batch_size = inputs.size(0)
        total_loss += loss.item() * batch_size
        
        _, predicted = outputs.max(1)
        total += batch_size
        correct += predicted.eq(targets).sum().item()
        
        pbar.set_postfix({"Loss": f"{total_loss / total:.4f}", "Acc": f"{(correct / total) * 100:.2f}%"})

    return total_loss / total, correct / total

def validate(model, loader, criterion, device, epoch):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc=f"Epoch {epoch+1:03d} [Val]  ", leave=False, disable=True)
    num_layers = len(model.layers)
    
    with torch.no_grad():
        for inputs, targets in pbar:
            inputs = inputs.to(device)
            targets = targets.squeeze().long().to(device)
            outputs, aux_loss = model(inputs)
            
            cls_loss = criterion(outputs, targets)
            scaled_aux_loss = (aux_loss / num_layers) * 0.01
            loss = cls_loss + scaled_aux_loss

            total_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            pbar.set_postfix({"Loss": f"{total_loss/total:.4f}", "Acc": f"{(correct/total)*100:.2f}%"})

    return total_loss / total, correct / total

train_losses, val_losses = [], []
train_accs, val_accs = [], []

print("\n--- Starting Training ---")
for epoch in range(epochs):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, epoch)
    val_loss, val_acc = validate(model, val_loader, criterion, device, epoch)
    scheduler.step()

    train_losses.append(tr_loss)
    val_losses.append(val_loss)
    train_accs.append(tr_acc)
    val_accs.append(val_acc)

    print(f"Epoch [{epoch+1:03d}/{epochs}] | Train Loss: {tr_loss:.4f} Acc: {tr_acc*100:.2f}% | Val Loss: {val_loss:.4f} Acc: {val_acc*100:.2f}%")

# ---------------------------------------------------------
# 5. Final Evaluation & Visualization Artifacts
# ---------------------------------------------------------
def generate_report_artifacts(model, loader, device, num_classes, class_labels):
    model.eval()
    y_score, y_true = [], []
    
    with torch.no_grad():
        for inputs, targets in tqdm(loader, desc="Testing"):
            inputs = inputs.to(device)
            outputs, _ = model(inputs)
            outputs = outputs.softmax(dim=-1)
            y_score.append(outputs.cpu().numpy())
            y_true.append(targets.squeeze().numpy())
            
    y_score = np.concatenate(y_score, axis=0)
    y_true = np.concatenate(y_true, axis=0)
    y_pred = np.argmax(y_score, axis=1)

    data_dir = os.path.dirname(os.path.abspath(data_npz_path))
    evaluator = Evaluator(data_flag, 'test', root=data_dir)
    medmnist_auc, medmnist_acc = evaluator.evaluate(y_score)
    
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    sensitivity = recall_score(y_true, y_pred, average='macro')
    f1 = f1_score(y_true, y_pred, average='macro')

    cm = confusion_matrix(y_true, y_pred)
    specificities = [
        (np.sum(cm) - (cm[i, i] + (np.sum(cm[i, :]) - cm[i, i]) + (np.sum(cm[:, i]) - cm[i, i]))) /
        ((np.sum(cm) - (cm[i, i] + (np.sum(cm[i, :]) - cm[i, i]) + (np.sum(cm[:, i]) - cm[i, i]))) + (np.sum(cm[:, i]) - cm[i, i]))
        for i in range(num_classes)
    ]
    specificity = np.mean(specificities)

    print("\n" + "="*50)
    print("        COMPREHENSIVE TEST EVALUATION REPORT     ")
    print("="*50)
    print(f"Overall Accuracy (OA / ACC) : {medmnist_acc * 100:.2f}%")
    print(f"Area Under Curve (AUC)      : {medmnist_auc:.4f}")
    print(f"Precision (Macro)            : {precision * 100:.2f}%")
    print(f"Sensitivity / Recall (Macro) : {sensitivity * 100:.2f}%")
    print(f"Specificity (Macro)          : {specificity * 100:.2f}%")
    print(f"F1-Score (Macro)             : {f1 * 100:.2f}%")
    print("="*50)
    
    print("\nClass-Wise Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=list(class_labels.values()), digits=4))

    # Saved Visualizations
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(1, epochs + 1), train_losses, label='Train Loss')
    plt.plot(range(1, epochs + 1), val_losses, label='Val Loss')
    plt.title('Loss Progression')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(range(1, epochs + 1), [a * 100 for a in train_accs], label='Train Acc')
    plt.plot(range(1, epochs + 1), [a * 100 for a in val_accs], label='Val Acc')
    plt.title('Accuracy Progression (%)')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('learning_curves.png', dpi=300)
    plt.close()

    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=list(class_labels.values()), yticklabels=list(class_labels.values()))
    plt.title('Test Set Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=300)
    plt.close()

    plt.figure(figsize=(9, 7))
    for i in range(num_classes):
        fpr, tpr, _ = roc_curve(y_true == i, y_score[:, i])
        plt.plot(fpr, tpr, label=f"{class_labels[str(i)]} (AUC = {auc(fpr, tpr):.3f})")
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True True Rate')
    plt.title('Multi-Class ROC Curves (PathMNIST)')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('roc_curves.png', dpi=300)
    plt.close()

    print("\nVisualizations saved successfully to Kaggle working directory:")
    print("  1. learning_curves.png")
    print("  2. confusion_matrix.png")
    print("  3. roc_curves.png\n")

print("\n--- Generating Final Evaluation & Visualizations ---")
generate_report_artifacts(model, test_loader, device, num_classes, info['label'])

Loading data from: pathmnist.npz
Executing on Device: cuda



--- Starting Training ---


Epoch [001/100] | Train Loss: 478.8247 Acc: 26.05% | Val Loss: 474.1288 Acc: 27.68%


Epoch [002/100] | Train Loss: 464.9741 Acc: 31.29% | Val Loss: 468.1521 Acc: 33.83%


Epoch [003/100] | Train Loss: 464.7062 Acc: 33.22% | Val Loss: 478.9986 Acc: 31.74%


Epoch [004/100] | Train Loss: 465.3249 Acc: 35.00% | Val Loss: 480.7958 Acc: 36.15%


... (124 outputs left out)

Testing:  97%|█████████▋| 28/29 [00:09<00:00,  3.94it/s]

Testing: 100%|██████████| 29/29 [00:09<00:00,  3.07it/s]


        COMPREHENSIVE TEST EVALUATION REPORT     
Overall Accuracy (OA / ACC) : 83.25%
Area Under Curve (AUC)      : 0.9747
Precision (Macro)            : 78.26%
Sensitivity / Recall (Macro) : 78.13%
Specificity (Macro)          : 97.93%
F1-Score (Macro)             : 77.85%

Class-Wise Classification Report:

                                      precision    recall  f1-score   support

                             adipose     0.9653    0.9552    0.9602      1338
                          background     0.9328    1.0000    0.9652       847
                              debris     0.4454    0.6372    0.5243       339
                         lymphocytes     0.9194    0.8817    0.9002       634
                               mucus     0.9759    0.8618    0.9153      1035
                       smooth muscle     0.7365    0.5997    0.6611       592
                 normal colon mucosa     0.8014    0.7571    0.7786       741
            cancer-associated stroma     0.4615    0.4703    0


Visualizations saved successfully to Kaggle working directory:
  1. learning_curves.png
  2. confusion_matrix.png
  3. roc_curves.png



In [3]:
# ---------------------------------------------------------
# 6. FGSM Adversarial Attack Suite
# ---------------------------------------------------------
def fgsm_attack(image, epsilon, data_grad):
    """
    Generates adversarial image using FGSM: x_adv = x + epsilon * sign(grad)
    Clips output to valid normalized image range [-1, 1].
    """
    sign_data_grad = data_grad.sign()
    perturbed_image = image + epsilon * sign_data_grad
    # Clip to maintain dataset normalization range [-1, 1]
    perturbed_image = torch.clamp(perturbed_image, -1.0, 1.0)
    return perturbed_image

def evaluate_fgsm(model, loader, criterion, device, epsilon):
    """Evaluates model accuracy under FGSM attack for a given epsilon."""
    model.eval()
    correct = 0
    total = 0
    
    for inputs, targets in tqdm(loader, desc=f"FGSM (eps={epsilon})", leave=False):
        inputs = inputs.to(device)
        targets = targets.squeeze().long().to(device)
        
        inputs.requires_grad = True
        
        # Forward pass
        outputs, _ = model(inputs)
        loss = criterion(outputs, targets)
        
        model.zero_grad()
        loss.backward()
        
        # Collect gradient and construct adversarial samples
        data_grad = inputs.grad.data
        perturbed_inputs = fgsm_attack(inputs, epsilon, data_grad)
        
        # Re-classify perturbed images
        with torch.no_grad():
            outputs_adv, _ = model(perturbed_inputs)
            _, predicted = outputs_adv.max(1)
            
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
    return (correct / total) * 100.0

def run_adversarial_suite(model, val_loader, test_loader, criterion, device, class_labels, epsilons=[0.0, 0.01, 0.03, 0.05, 0.1, 0.15, 0.2]):
    print("\n" + "="*50)
    print("        RUNNING FGSM ADVERSARIAL BENCHMARK       ")
    print("="*50)
    
    val_accuracies = []
    test_accuracies = []
    
    for eps in epsilons:
        val_acc = evaluate_fgsm(model, val_loader, criterion, device, eps)
        test_acc = evaluate_fgsm(model, test_loader, criterion, device, eps)
        val_accuracies.append(val_acc)
        test_accuracies.append(test_acc)
        print(f"Epsilon: {eps:<5} | Val Acc: {val_acc:.2f}% | Test Acc: {test_acc:.2f}%")

    # ---------------------------------------------------------
    # Plot 1: Adversarial Accuracy vs. Epsilon Plot (For Report)
    # ---------------------------------------------------------
    plt.figure(figsize=(9, 6))
    plt.plot(epsilons, val_accuracies, 'o-', color='crimson', linewidth=2, label='Validation Accuracy')
    plt.plot(epsilons, test_accuracies, 's--', color='royalblue', linewidth=2, label='Test Accuracy')
    plt.title('Robustness Breakdown: Model Accuracy vs. FGSM Perturbation (Epsilon)')
    plt.xlabel(r'Adversarial Perturbation Magnitude (Epsilon $\epsilon$)')
    plt.ylabel('Accuracy (%)')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.savefig('fgsm_accuracy_vs_epsilon.png', dpi=300)
    plt.close()
    print("\nSaved curve graph: fgsm_accuracy_vs_epsilon.png")

    # ---------------------------------------------------------
    # Plot 2: Visual Comparison Grid (Clean vs Perturbation vs Attacked)
    # ---------------------------------------------------------
    print("Generating qualitative adversarial visual examples...")
    target_eps = 0.05  # Moderate epsilon for clear visual comparison
    model.eval()
    
    sample_inputs, sample_targets = next(iter(test_loader))
    sample_inputs = sample_inputs.to(device)
    sample_targets = sample_targets.squeeze().long().to(device)
    sample_inputs.requires_grad = True

    outputs, _ = model(sample_inputs)
    loss = criterion(outputs, sample_targets)
    model.zero_grad()
    loss.backward()

    data_grad = sample_inputs.grad.data
    perturbed_inputs = fgsm_attack(sample_inputs, target_eps, data_grad)

    with torch.no_grad():
        outputs_clean, _ = model(sample_inputs)
        outputs_adv, _ = model(perturbed_inputs)
        
        preds_clean = outputs_clean.argmax(dim=1).cpu().numpy()
        preds_adv = outputs_adv.argmax(dim=1).cpu().numpy()

    # Move tensors back to CPU for plotting & un-normalize [-1, 1] -> [0, 1]
    inputs_np = (sample_inputs.cpu().detach().numpy().transpose(0, 2, 3, 1) + 1.0) / 2.0
    perturbed_np = (perturbed_inputs.cpu().detach().numpy().transpose(0, 2, 3, 1) + 1.0) / 2.0
    grad_np = (data_grad.sign().cpu().numpy().transpose(0, 2, 3, 1) + 1.0) / 2.0

    targets_np = sample_targets.cpu().numpy()

    num_samples = 4
    fig, axes = plt.subplots(num_samples, 3, figsize=(11, 3.5 * num_samples))
    
    for i in range(num_samples):
        true_lbl = class_labels[str(targets_np[i])]
        clean_lbl = class_labels[str(preds_clean[i])]
        adv_lbl = class_labels[str(preds_adv[i])]

        # Column 1: Clean
        axes[i, 0].imshow(np.clip(inputs_np[i], 0, 1))
        axes[i, 0].set_title(f"Original Sample\nTrue: {true_lbl}\nPred: {clean_lbl}", color='green' if true_lbl == clean_lbl else 'red')
        axes[i, 0].axis('off')

        # Column 2: Perturbation Noise
        axes[i, 1].imshow(np.clip(grad_np[i], 0, 1))
        axes[i, 1].set_title(rf"FGSM Noise Pattern\n($\epsilon = {target_eps}$)")
        axes[i, 1].axis('off')

        # Column 3: Adversarial
        axes[i, 2].imshow(np.clip(perturbed_np[i], 0, 1))
        axes[i, 2].set_title(f"Attacked Sample\nTrue: {true_lbl}\nAttacked Pred: {adv_lbl}", color='green' if true_lbl == adv_lbl else 'red')
        axes[i, 2].axis('off')

    plt.tight_layout()
    plt.savefig('fgsm_visual_samples.png', dpi=300)
    plt.close()
    print("Saved sample grid visualization: fgsm_visual_samples.png\n")

# ---------------------------------------------------------
# Run FGSM Benchmark Suite
# ---------------------------------------------------------
run_adversarial_suite(model, val_loader, test_loader, criterion, device, info['label'])


        RUNNING FGSM ADVERSARIAL BENCHMARK       


FGSM (eps=0.0):   0%|          | 0/40 [00:00<?, ?it/s]

FGSM (eps=0.0):   2%|▎         | 1/40 [00:01<00:57,  1.46s/it]

FGSM (eps=0.0):   5%|▌         | 2/40 [00:02<00:44,  1.17s/it]

FGSM (eps=0.0):   8%|▊         | 3/40 [00:03<00:39,  1.08s/it]

FGSM (eps=0.0):  10%|█         | 4/40 [00:04<00:37,  1.03s/it]

... (497 outputs left out)

FGSM (eps=0.2): 100%|██████████| 29/29 [00:27<00:00,  1.41it/s]

Epsilon: 0.2   | Val Acc: 50.72% | Test Acc: 40.57%



Saved curve graph: fgsm_accuracy_vs_epsilon.png
Generating qualitative adversarial visual examples...


Saved sample grid visualization: fgsm_visual_samples.png



In [4]:
# ---------------------------------------------------------
# 6. FLOPs & MACs Benchmark Section
# ---------------------------------------------------------
from fvcore.nn import FlopCountAnalysis

# Wrapper class so fvcore only evaluates the image prediction forward pass (logits)
class ModelWrapper(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, x):
        logits, _ = self.base_model(x)
        return logits

model.eval()
wrapped_model = ModelWrapper(model).to(device)

# Measure FLOPs per image using the first test sample
sample_images, _ = next(iter(test_loader))
sample_single_image = sample_images[0:1].to(device)

flops_analysis = FlopCountAnalysis(wrapped_model, sample_single_image)
flops_per_image = flops_analysis.total()
total_test_samples = len(test_dataset)

# Total inference cost across all test samples
total_test_flops = flops_per_image * total_test_samples
total_test_macs = total_test_flops / 2

print("\n" + "="*50)
print("     FULL TEST SET FLOPs BENCHMARK REPORT     ")
print("="*50)
print(f"Total Test Samples          : {total_test_samples}")
print(f"FLOPs Per Single Image      : {flops_per_image / 1e9:.4f} GFLOPs")
print(f"MACs Per Single Image       : {flops_per_image / (2 * 1e9):.4f} GMACs")
print("-" * 50)
print(f"Entire Test Set Total FLOPs : {total_test_flops / 1e9:.4f} GFLOPs")
print(f"Entire Test Set Total MACs  : {total_test_macs / 1e9:.4f} GMACs")
print("="*50)

/scratch/data/p26ai0007/iitj/ml_project/.venv/lib/python3.10/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Unsupported operator aten::add encountered 19 time(s)


Unsupported operator aten::repeat_interleave encountered 18 time(s)


Unsupported operator aten::div encountered 9 time(s)


Unsupported operator aten::softmax encountered 18 time(s)


Unsupported operator aten::topk encountered 9 time(s)


... (2 outputs left out)

Unsupported operator aten::mul encountered 68 time(s)


Unsupported operator aten::add_ encountered 34 time(s)


The following submodules of the model were never called during the trace of the graph. They may be unused, or they were accessed by direct calls to .forward() or via other python methods. In the latter case they will have zeros for statistics, though their statistics will still contribute to their parent calling module.
base_model.layers.8.moe.experts.2, base_model.layers.8.moe.experts.2.v, base_model.layers.8.moe.experts.2.w1, base_model.layers.8.moe.experts.2.w2, base_model.layers.8.moe.experts.4, base_model.layers.8.moe.experts.4.v, base_model.layers.8.moe.experts.4.w1, base_model.layers.8.moe.experts.4.w2



     FULL TEST SET FLOPs BENCHMARK REPORT     
Total Test Samples          : 7180
FLOPs Per Single Image      : 0.5610 GFLOPs
MACs Per Single Image       : 0.2805 GMACs
--------------------------------------------------
Entire Test Set Total FLOPs : 4028.0727 GFLOPs
Entire Test Set Total MACs  : 2014.0364 GMACs
